In [ ]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

# Video segmentation with SAM 2 — one frame at a time

Point at something in a video frame, and SAM 2 follows it for the rest of the clip. This notebook covers:

1. **Clicking** to select an object, and clicking again to fix a selection that grabbed the wrong thing
2. **Propagating** that selection through the video to get a *masklet* — the mask on every frame
3. **Tracking several objects at once**, each under its own id
4. **Boxing** an object instead of clicking it
5. **Bounding the memory**, so long videos do not grow without limit
6. **Swapping in EfficientTAM**, a lighter model, through the same code

*Mask* means the prediction for one object on one frame; *masklet* means that mask across the whole clip.

### Why this fork streams

The original SAM 2 video predictor loads **every frame of the video into memory** before it starts tracking. A long clip needs a lot of RAM, and a live camera feed cannot be handled at all.

Here you hand the model **one frame per call**. The only thing kept between frames is a small per-object *memory bank*, which you can cap — so memory stays roughly flat whether the clip is 200 frames or 200,000, and the same code runs on a live stream.

<a target="_blank" href="https://colab.research.google.com/github/cjaverliat/sam2/blob/main/notebooks/video_predictor_example.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Set-up

Running locally? Install the package first — see the [installation instructions](https://github.com/cjaverliat/sam2#installation).

Running on Colab? Set `using_colab = True` below and run the next cell. Pick a GPU runtime under *Edit → Notebook Settings → Hardware accelerator*.

In [ ]:
using_colab = False

In [ ]:
if using_colab:
    import sys

    !{sys.executable} -m pip install opencv-python matplotlib
    !{sys.executable} -m pip install 'git+https://github.com/cjaverliat/sam2.git'

    # the example clip, the shared helpers, and the checkpoint this notebook loads
    !mkdir -p videos
    !wget -P videos https://raw.githubusercontent.com/cjaverliat/sam2/main/notebooks/videos/bedroom.mp4
    !wget https://raw.githubusercontent.com/cjaverliat/sam2/main/notebooks/nb_utils.py

    !mkdir -p ../checkpoints/
    !wget -P ../checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt

In [ ]:
import os
import sys

# if using Apple MPS, fall back to CPU for unsupported ops
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm

if os.path.isdir("notebooks"):     # started from the repo root rather than notebooks/
    sys.path.insert(0, "notebooks")
import nb_utils as nb  # shared plotting helpers — see notebooks/nb_utils.py

nb.use_dark(False)  # light figures; see nb_utils.use_dark for PyCharm's inversion
device = nb.pick_device()

### Load the model

`build_sam2_video_predictor` gives you the streaming predictor. Loading takes a few seconds. `base_plus` is a good default; `tiny`, `small` and `large` are the other sizes, and the same code runs on all of them.

In [ ]:
from sam.build_sam import build_sam2_video_predictor

CHECKPOINT = "../checkpoints/sam2.1_hiera_base_plus.pt"
MODEL_CFG = "configs/sam2.1/sam2.1_hiera_b+.yaml"

nb.require_checkpoints((CHECKPOINT, "pixi run download-sam2-base-plus"))

predictor = build_sam2_video_predictor(
    MODEL_CFG, CHECKPOINT, device=device,
    vos_optimized=False, apply_postprocessing=True, use_half=True,
)
print("ready")

### The clip

`bedroom.mp4` is 200 frames of 960×540. We read it with OpenCV one frame at a time — which is the point: nothing is buffered, so the same loop works on a webcam or an RTSP stream.

A **session** holds one video's worth of state. It counts frames for you, so you feed it pixels and nothing else. It takes no video size — that comes from the first frame you hand it.

In [ ]:
VIDEO = "./videos/bedroom.mp4"


def open_clip(path=VIDEO):
    """Re-open the clip. Each demo below streams it from the start."""
    cap = cv2.VideoCapture(path)
    return cap, int(cap.get(cv2.CAP_PROP_FRAME_COUNT))


def read_frame(cap):
    """Next frame as a (C, H, W) float tensor in [0, 1], or None at the end."""
    ok, frame = cap.read()
    if not ok:
        return None
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return torch.as_tensor(frame).permute(2, 0, 1).to(device) / 255.0


def as_image(frame):
    """A (C, H, W) tensor back to something `imshow` accepts."""
    return frame.permute(1, 2, 0).cpu().numpy()


cap, n_frames = open_clip()
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"{n_frames} frames of {width}x{height}")

first_frame = read_frame(cap)
plt.figure(figsize=(7, 4))
plt.title("frame 0")
plt.imshow(as_image(first_frame))
plt.axis("off")
plt.show()

## 1. Click on an object

Let's segment the child on the left. One **positive click** at (x, y) = (210, 350) — positive meaning *this is part of what I want*. A negative click (`label=0`) means *not this part*.

Give the object an id of your choosing. Every result from here on comes back keyed by that id.

In [ ]:
from sam.prompts import GeometryPrompt

OBJ_ID = 1
CLICK = (210, 350)

session = predictor.start_session()
masklets = session.process(first_frame, prompts=[GeometryPrompt.click(OBJ_ID, CLICK)])

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_title("one click — the model picks an object")
ax.imshow(as_image(first_frame))
ax.axis("off")
nb.show_mask(nb.to_mask(masklets[OBJ_ID]), ax, obj_id=OBJ_ID)
nb.show_points([CLICK], [1], ax)
plt.show()

### Refining a selection

That click landed on the shorts, and the model had no way to know whether we meant the shorts or the whole child — one point is genuinely ambiguous. A second positive click on the shirt says *this too*.

Two things worth knowing:

- Send **all** the clicks, not just the new one. Each object takes one prompt per frame, so that prompt carries the full list of points. `GeometryPrompt.click` is shorthand for a single point; several points use the general `GeometryPrompt(...)` form.
- We are re-running a frame the session has already seen, so we pass `frame_idx=0`. The session's counter resumes from there.

In [ ]:
CLICKS = [[210, 350], [250, 220]]  # the original click, plus one on the shirt
LABELS = [1, 1]                    # both positive

prompt = GeometryPrompt(
    obj_id=OBJ_ID,
    points_coords=torch.tensor(CLICKS, dtype=torch.float32),
    points_labels=torch.tensor(LABELS),
)
masklets = session.process(first_frame, prompts=[prompt], frame_idx=0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_title("two clicks — the whole child")
ax.imshow(as_image(first_frame))
ax.axis("off")
nb.show_mask(nb.to_mask(masklets[OBJ_ID]), ax, obj_id=OBJ_ID)
nb.show_points(CLICKS, LABELS, ax)
plt.show()

## 2. Propagate through the video

Now just feed frames. No more prompts — the session carries the object forward on its own.

The progress bar reports what the memory bank holds:

- **conditional** memories come from frames you prompted. There is one, from frame 0, and it is never discarded.
- **non-conditional** memories come from frames the model tracked by itself.

Watch the second number climb by one per frame. That is the default bank, which keeps everything — faithful to the original paper, and fine for a 200-frame clip. Section 5 caps it.

In [ ]:
bank = session.state.memory_bank
frames, tracked = [first_frame], [{OBJ_ID: nb.to_mask(masklets[OBJ_ID])}]

pbar = tqdm(range(1, n_frames), desc="propagating")
for _ in pbar:
    frame = read_frame(cap)
    if frame is None:
        break
    masklets = session.process(frame)          # no prompts: just pixels
    frames.append(frame)
    tracked.append({i: nb.to_mask(m) for i, m in masklets.items()})
    pbar.set_postfix({
        "conditional": bank.count_conditional_memories(),
        "non_conditional": bank.count_non_conditional_memories(),
    })
pbar.close()
cap.release()

nb.show_frames([as_image(f) for f in frames], tracked,
               "one click on frame 0, tracked through the clip",
               idxs=[0, 60, 120, len(frames) - 1], label_prefix="child")

## 3. Several objects at once

Give each object its own id and send one prompt per object. They are tracked together in the same pass, and every call returns one mask per object.

Prompts are also not restricted to the first frame — below, the second child is added at frame 20, and is tracked from there on.

In [ ]:
# frame index -> the prompts to send on that frame
SCHEDULE = {
    0:  [GeometryPrompt.click(1, (210, 350))],   # the child on the left
    20: [GeometryPrompt.click(2, (385, 230))],   # a second child, added later
}
N = 120

cap, _ = open_clip()
multi_session = predictor.start_session()

frames, tracked = [], []
for i in tqdm(range(N), desc="two objects"):
    frame = read_frame(cap)
    if frame is None:
        break
    masklets = multi_session.process(frame, prompts=SCHEDULE.get(i))
    frames.append(frame)
    tracked.append({oid: nb.to_mask(m) for oid, m in masklets.items()})
cap.release()

print("ids on frame 10:", sorted(tracked[10]), "| ids on the last frame:", sorted(tracked[-1]))
nb.show_frames([as_image(f) for f in frames], tracked,
               "one child from frame 0, a second added at frame 20",
               idxs=[0, 19, 21, len(frames) - 1], label_prefix="child")

## 4. Box instead of click

If you already have a bounding box — from a detector, a tracker, or an annotation tool — use it directly. The box is encoded as its two corners and seeds exactly one object, just as a click would.

In [ ]:
BOX = (285, 0, 535, 430)  # xyxy pixels, around the girl
N = 120

cap, _ = open_clip()
box_session = predictor.start_session()

frames, tracked = [], []
for i in tqdm(range(N), desc="box prompt"):
    frame = read_frame(cap)
    if frame is None:
        break
    prompts = [GeometryPrompt.box(OBJ_ID, BOX)] if i == 0 else None
    masklets = box_session.process(frame, prompts=prompts)
    frames.append(frame)
    tracked.append({oid: nb.to_mask(m) for oid, m in masklets.items()})
cap.release()

nb.show_frames(
    [as_image(f) for f in frames], tracked,
    "boxed on frame 0, tracked from there",
    idxs=[0, 40, 80, len(frames) - 1], label_prefix="girl",
    extra=lambda ax, i: nb.show_box(BOX, ax, obj_id=5, label="prompt box", style="--")
    if i == 0 else None,
)

## 5. Bounding the memory

The default bank never forgets, so what it holds grows with the clip. `ForgetfulObjectMemoryBank` keeps a sliding window instead: prompted frames are kept forever, tracked frames older than the window are dropped. Peak memory then stops depending on clip length — which is what makes hour-long videos and live feeds practical.

A custom bank is set on the *state*, so this is the one place you build the state yourself and call `forward` instead of using a session. A session is a convenience wrapper over exactly this — one `forward` call per frame.

Run it and compare against what the default bank would have held.

In [ ]:
from sam.models.sam2_predictor import Sam2VideoPredictorState
from sam.modeling.memory.forgetful import ForgetfulObjectMemoryBank

WINDOW = 7
N = 120

cap, _ = open_clip()
state = Sam2VideoPredictorState.create(
    video_hw=(height, width),
    memory_bank=ForgetfulObjectMemoryBank(memory_window_size=WINDOW),
)

held = []
for i in tqdm(range(N), desc=f"forgetful bank (window={WINDOW})"):
    frame = read_frame(cap)
    if frame is None:
        break
    prompts = [GeometryPrompt.click(OBJ_ID, CLICK)] if i == 0 else []
    predictor.forward(state=state, frame_idx=i, frame=frame, prompts=prompts)
    held.append(state.memory_bank.count_non_conditional_memories())
cap.release()

plt.figure(figsize=(8, 3.2))
plt.plot(held, label=f"ForgetfulObjectMemoryBank(window={WINDOW})")
plt.plot(range(len(held)), "--", label="default bank (keeps everything)")
plt.xlabel("frame")
plt.ylabel("non-conditional memories held")
plt.title("bounded vs unbounded memory")
plt.legend()
plt.tight_layout()
plt.show()

print(f"after {len(held)} frames: forgetful holds {held[-1]}, the default would hold {len(held) - 1}")

Want a different policy — a cap on the count, eviction by score, temporal striding? Subclass [`ObjectMemoryBank`](../sam/modeling/memory/bank.py) and override `try_add_memories`, `select_memories` and `prune_memories`.

## 6. The same code, a lighter model

[EfficientTAM](https://github.com/yformer/EfficientTAM) is a smaller Track-Anything model that plugs into the same predictor — point the builder at a different config and checkpoint, and nothing else changes. Useful when SAM 2 is heavier than your hardware wants.

In [ ]:
ETAM_CHECKPOINT = "../checkpoints/efficienttam_ti.pt"
ETAM_CFG = "configs/efficienttam/efficienttam_ti.yaml"
N = 60

nb.require_checkpoints((ETAM_CHECKPOINT, "pixi run download-efficienttam"))
etam = build_sam2_video_predictor(ETAM_CFG, ETAM_CHECKPOINT, device=device, use_half=True)

cap, _ = open_clip()
etam_session = etam.start_session()

frames, tracked = [], []
for i in tqdm(range(N), desc="EfficientTAM"):
    frame = read_frame(cap)
    if frame is None:
        break
    prompts = [GeometryPrompt.click(OBJ_ID, CLICK)] if i == 0 else None
    masklets = etam_session.process(frame, prompts=prompts)
    frames.append(frame)
    tracked.append({oid: nb.to_mask(m) for oid, m in masklets.items()})
cap.release()

m = lambda p: sum(x.numel() for x in p.parameters()) / 1e6
print(f"SAM 2 base_plus: {m(predictor):.0f}M params | EfficientTAM tiny: {m(etam):.0f}M params")

nb.show_frames([as_image(f) for f in frames], tracked,
               "EfficientTAM tiny — identical API",
               idxs=[0, 20, 40, len(frames) - 1], label_prefix="child")

## What you can do from here

| | how |
|---|---|
| Select an object | `GeometryPrompt.click(obj_id, (x, y))`, `label=0` to subtract |
| Refine it | all points in one `GeometryPrompt` for that id, re-running the frame with `frame_idx=` |
| Start from a box | `GeometryPrompt.box(obj_id, (x0, y0, x1, y1))` |
| Track many objects | one prompt per id; results come back keyed by id |
| Add an object later | prompt on any frame, not only the first |
| Keep memory flat | `ForgetfulObjectMemoryBank(memory_window_size=...)` on the state |
| Go lighter | EfficientTAM configs, same builder, same API |
| Segment by *description* | that is SAM 3 — see [`sam3_video_predictor_example.ipynb`](./sam3_video_predictor_example.ipynb) |

Every result carries `best_mask_logits`, where positive means foreground — so `logits > 0` is your binary mask, which is all `nb.to_mask` does.

Two things SAM 2 cannot do: it has no idea what objects *are*, so it cannot find one from a description, and it cannot notice a new object arriving on its own. You point, it follows. SAM 3 adds both.